# M3 KNN — 실습 (W5)

> 위에서부터 한 셀씩 `Shift+Enter`로 실행하세요. `___` 빈칸은 직접 채웁니다.

**이 실습이 끝나면**
1. 6점 세트피스의 **제곱거리·다수결**을 numpy로 재현하고(k=1 낚임 → k=3 역전) **sklearn 검산**으로 일치를 확인한다 ⭐
2. **단위 손계산**(m→cm에 이웃 교체)과 와인 실측(**0.722 → 0.944**)으로 스케일링의 필요성을 몸으로 안다 ⭐
3. `Pipeline`으로 누수 없이, **교차검증**(M2b 첫 출동)으로 k를 고르고(k=12 → test 0.963), 결정경계로 과적합을 눈으로 본다

**7단계 멘탈모델 초점:** 표현(거리·스케일) + 모델

## Part A. 세트피스 — 세상에서 가장 작은 KNN ⭐
훈련 6점: A(3,3),(2,4),(5,2) / B(4,5)←**A 지역에 낀 잡음**, (9,9),(10,8). 질의 ★(4,4). 먼저 종이에서 제곱거리 표(√ 생략!)를 만들고 k=1/3/5를 판정해 본 뒤, 코드로 재현하세요.

In [ ]:
import numpy as np                                     # 수치 계산

Xm = np.array([[3, 3], [2, 4], [5, 2],                 # A 클래스 3점
               [4, 5], [9, 9], [10, 8]], dtype=float)  # B 클래스 3점((4,5)는 잡음)
ym = np.array([0, 0, 0, 1, 1, 1])                      # 0=A, 1=B
q = np.array([4, 4], dtype=float)                      # 질의점 ★

d2 = ((Xm - q) ** 2).sum(axis=___)                     # ✍️ 빈칸: 제곱거리 — 특징 축 방향으로 합(√ 생략 트릭)
print('제곱거리:', d2.tolist())                          # [2, 4, 5, 1, 50, 52] — 1등이 잡음!
order = np.argsort(d2)                                 # 가까운 순 정렬
for k in (1, 3, 5):                                    # k를 바꿔가며
    votes = ym[order[:k]]                              # 이웃 k개의 라벨
    pred = np.bincount(votes).___()                    # ✍️ 빈칸: 다수결 — 표가 가장 많은 라벨
    print(f'k={k}: 이웃 {votes.tolist()} → 예측', 'A' if pred == 0 else 'B')

In [ ]:
from sklearn.neighbors import KNeighborsClassifier     # 이제 sklearn으로 검산

for k in (1, 3, 5):                                    # 같은 6점, 같은 질의
    m = KNeighborsClassifier(n_neighbors=k).fit(Xm, ym)
    print(f'sklearn k={k}:', 'A' if m.predict([q])[0] == 0 else 'B')  # 손계산과 같은가?

> **검산 포인트:** 제곱거리 [2, 4, 5, **1**, 50, 52] — 1등이 하필 잡음 B(4,5). **k=1 → B(낚임!)**, k=3 → {B,A,A} 2:1 → **A**, k=5 → 3:2 → A. sklearn 판정도 동일. **k=1 = 잡음 한 점에 휘둘리는 과적합의 기하학적 얼굴**, 다수결 = 잡음을 눌러 담는 장치. (√ 없이 제곱거리로 순위 비교 — 실전 트릭.)

## Part B. 단위 손계산 — m vs cm에 이웃이 뒤바뀐다 ⭐
A(1.60m, 60kg), B(1.80m, 59kg), 질의 q(1.79m, 60kg). 상식적 이웃은 키가 거의 같은 B — 정말 그렇게 나올까요?

In [ ]:
A_m = np.array([1.60, 60.0])                           # 키(m), 몸무게(kg)
B_m = np.array([1.80, 59.0])
q_m = np.array([1.79, 60.0])
print('m 표기 : d²(A) =', round(((q_m - A_m) ** 2).sum(), 4),
      '| d²(B) =', round(((q_m - B_m) ** 2).sum(), 4))  # 0.0361 vs 1.0001 → A?! (몸무게 지배)

A_cm, B_cm, q_cm = A_m.copy(), B_m.copy(), q_m.copy()  # 같은 사람들
A_cm[0] *= ___                                         # ✍️ 빈칸: 키를 cm로 — 몇을 곱하나?
B_cm[0] *= 100
q_cm[0] *= 100
print('cm 표기: d²(A) =', round(((q_cm - A_cm) ** 2).sum(), 4),
      '| d²(B) =', round(((q_cm - B_cm) ** 2).sum(), 4))  # 361 vs 2 → B (키 지배)

> **관찰:** m 표기에선 이웃이 **A**(몸무게 1kg 차이가 키 0.19m를 압도), cm 표기에선 **B**(키가 지배). **같은 세 사람인데 표기만 바꿨더니 이웃이 뒤바뀜** — 거리는 물리적 의미가 아니라 숫자의 크기에 휘둘립니다. 해결 = 스케일링(발언권 통일). 다음 Part에서 실전 데이터로.

## Part C. 와인 실측 — 스케일링 전후, 그리고 Pipeline
와인 13특징(proline 0~1700, hue 0~2 — 발언권이 하늘과 땅)에서 스케일링의 효과를 재고, 누수 없는 배관(`Pipeline`)으로 같은 일을 안전하게 합니다.

In [ ]:
import matplotlib.pyplot as plt                        # 그래프
from sklearn.datasets import load_wine                 # 와인 데이터(3품종)
from sklearn.model_selection import train_test_split   # 공정한 시험(M2a)
from sklearn.preprocessing import StandardScaler       # 표준화
from sklearn.pipeline import make_pipeline             # 누수 없는 배관

wine = load_wine()                                     # 178병, 13특징
X, y = wine.data, wine.target
X_train, X_test, y_train, y_test = train_test_split(   # 30% 시험, 비율 유지
    X, y, test_size=0.3, random_state=42, stratify=y)

raw = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)      # 스케일링 없이
print('스케일링 전:', round(raw.score(X_test, y_test), 3))            # 0.722

scaler = StandardScaler()                              # 표준화 도구
X_train_s = scaler.fit_transform(X_train)              # train에 fit + 변환(기준 학습)
X_test_s = scaler.transform(___)                       # ✍️ 빈칸: test는 같은 기준으로 '변환만'(fit 금지 — 누수!)
knn_s = KNeighborsClassifier(n_neighbors=5).fit(X_train_s, y_train)  # 같은 모델·같은 k
print('스케일링 후:', round(knn_s.score(X_test_s, y_test), 3))        # 0.944 — +22.2%p!

pipe = make_pipeline(StandardScaler(),                 # 같은 일을 자동으로
                     KNeighborsClassifier(n_neighbors=5))
pipe.fit(X_train, y_train)                             # 스케일러는 train에만 fit됨
print('Pipeline  :', round(pipe.score(X_test, y_test), 3))            # 0.944 — 동일

> **관찰:** **0.722 → 0.944** — 모델·k 그대로, 스케일링만으로 +22.2%p(Part B의 단위 문제가 13차원에서 벌어지고 있던 것). test에 `fit_transform`이 아니라 **`transform`** 인 이유 = M2a의 누수 원칙(train에서 배운 기준을 적용만). `Pipeline`은 이 순서를 자동 보장 — 특히 다음 Part의 교차검증과 결합할 때 진가.

## Part D. k 선택 — 교차검증의 첫 출동 (M2b)
k=1의 train 점수를 먼저 보고(왜 만점일까요?), **train만으로 5-fold CV**를 돌려 k를 고른 뒤, 봉인된 test를 마지막 한 번 엽니다. "test를 보며 고르면" 어떻게 되는지도 비교합니다.

In [ ]:
from sklearn.model_selection import cross_val_score    # 교차검증(M2b)

k1 = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=1)).fit(X_train, y_train)
print('k=1: train', round(k1.score(X_train, y_train), 3),  # 1.0 — 자기 자신이 이웃(만점 = 경고!)
      '/ test', round(k1.score(X_test, y_test), 3))

ks = range(1, 21)                                      # k 후보 1~20
cv_means = []                                          # CV 평균 저장
for k in ks:                                           # 각 k에 대해
    p = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))  # 누수 없는 배관째로
    scores = cross_val_score(p, X_train, y_train, cv=___)  # ✍️ 빈칸: 몇 조각 교차검증?(M2b의 그 수)
    cv_means.append(scores.mean())                     # 평균 저장

plt.plot(list(ks), cv_means, 'o-')                     # k별 CV 곡선
plt.xlabel('k (number of neighbors)'); plt.ylabel('CV mean accuracy (train only)')  # 축(영어)
plt.grid(True, alpha=0.3)
plt.title('Choosing k by cross-validation')
plt.show()

best_k = list(ks)[int(np.argmax(cv_means))]            # CV가 고른 k
print('CV가 고른 k:', best_k, '| CV 평균:', round(max(cv_means), 3))   # 12 / 0.968
final = make_pipeline(StandardScaler(),
                      KNeighborsClassifier(n_neighbors=___))  # ✍️ 빈칸: CV가 고른 그 k(위 출력의 수)
final.fit(X_train, y_train)                            # 최종 모델
print('봉인 해제 — 최종 test(한 번):', round(final.score(X_test, y_test), 3))  # 0.963 — 정직한 숫자

test_accs = [make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
             .fit(X_train, y_train).score(X_test, y_test) for k in ks]  # (비교용) test를 보며 고르면?
print('(비교) test로 고른 k:', int(np.argmax(test_accs)) + 1,
      '| 그 점수:', round(max(test_accs), 3), '← 부풀려진 숫자(완만한 누수)')

> **관찰:** k=1은 train **1.0**(자기 자신이 최근접 이웃 — **train 만점은 경고**, M2a). CV(train만, 5-fold)가 고른 k=**12** → 봉인 해제 test **0.963**. test를 보며 고르면 k=13에 0.981이 나와 "더 좋아 보이지만" 그건 **시험지를 보며 고른 점수**(완만한 누수) — 0.963이 정직한 보고값. M2b의 도구가 실전에서 처음 일한 순간입니다.

## Part E. 결정경계 — k=1 vs k=15를 눈으로
2개 특징(alcohol, proline)만 표준화해 결정경계를 그립니다. k=1의 "들쭉날쭉 + 고립된 섬"과 k=15의 매끄러움을 비교하세요.

In [ ]:
from matplotlib.colors import ListedColormap           # 색 지정

X2 = StandardScaler().fit_transform(X[:, [0, 12]])     # alcohol, proline 2특징(시각화용 전체)
xx, yy = np.meshgrid(np.linspace(X2[:, 0].min() - 0.5, X2[:, 0].max() + 0.5, 300),
                     np.linspace(X2[:, 1].min() - 0.5, X2[:, 1].max() + 0.5, 300))  # 촘촘한 격자

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, k in zip(axes, (___, 15)):                     # ✍️ 빈칸: 과적합 쪽 k(잡음 하나하나를 감싸는 최솟값)
    clf = KNeighborsClassifier(n_neighbors=k).fit(X2, y)  # 2D에서 학습
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)  # 격자점 전부 예측
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=ListedColormap(['#fca5a5', '#86efac', '#93c5fd']))
    ax.scatter(X2[:, 0], X2[:, 1], c=y, edgecolor='k', s=18,
               cmap=ListedColormap(['#dc2626', '#16a34a', '#2563eb']))
    ax.set_xlabel('alcohol (scaled)'); ax.set_ylabel('proline (scaled)')  # 축(영어)
    ax.set_title(f'KNN decision boundary (k={k})')
plt.tight_layout(); plt.show()

> **관찰:** k=1은 잡음 한 점 한 점을 감싸는 **들쭉날쭉한 경계와 고립된 섬들**(암기의 지도), k=15는 동네의 대세를 따르는 **매끄러운 경계**. M2a 깊이 스윕의 기하학 버전 — **복잡한 경계 = 과적합 위험.** Part A 세트피스의 "잡음에 낚이는 k=1"이 실제 데이터에서는 이런 모습입니다.

## 🤖 AI 코파일럿 활용 (선택) — ai-native v1
막히면 AI 튜터에게 묻되, **먼저 스스로 생각**하고 답을 **실행으로 검증**하세요.

**좋은 질문 예시**
- "6점 세트피스의 제곱거리 표를 내가 만들 테니 채점해 줘."
- "k=1의 train 정확도가 왜 항상 1.0인지 설명해 볼게 — 허점을 찔러 줘."
- "m/cm 손계산으로 스케일링의 필요성을 설명해 볼게."
- "test로 k를 고른 0.981이 왜 부풀려진 숫자인지 M2a로 설명해 볼게."

**가드레일**
1. 먼저 손으로 생각 → 그 다음 AI
2. AI 코드는 *왜 그런지* 설명할 수 있을 때만 사용
3. AI 출력은 실행으로 검증

## 정리 & 자가 점검

**오늘 한 일 3줄**
1. 6점 세트피스를 손계산·numpy·sklearn 삼중으로 완주했다 — k=1은 잡음에 낚이고(과적합의 기하학) k=3은 다수결로 역전
2. 단위 손계산(m→cm에 이웃 교체)과 와인 실측(0.722 → 0.944)으로 **스케일링이 생명**임을 확인하고 `Pipeline`으로 누수 없이 처리했다
3. **교차검증**(M2b 첫 출동)으로 k=12를 골라 test 0.963(정직) — test로 고른 0.981(부풀려짐)과 대조했다

**스스로 점검**
- [ ] 제곱거리 표를 만들고 k=1/3/5 판정을 할 수 있다
- [ ] k=1의 train이 왜 1.0인지, 그게 왜 경고인지 안다
- [ ] 스케일러를 train에만 fit하는 이유와 Pipeline의 역할을 안다
- [ ] k를 CV로 고르는 절차(그리고 test로 고르면 안 되는 이유)를 안다
- [ ] 차원의 저주가 KNN에 치명적인 이유를 말할 수 있다

**🔹심화 (선택)**
- `weights='distance'`(가까운 이웃에 더 큰 표)로 Part C를 재실행 — 이 데이터에선 얼마나 달라지나요? (실측: 0.944로 차이 없음)
- Part A의 질의점을 (5,4), (7,7)로 옮겨 보세요 — k=1과 k=3의 판정이 언제 갈리나요?
- `KNeighborsRegressor`로 이웃 평균 회귀를 맛보세요 — 다수결의 회귀판.

**다음 시간(M4):** 외우지 않고 **직선 하나로 요약**하는 선형 회귀 — 그 직선을 찾는 원리(손실을 줄이는 방향으로 조금씩)가 2학기 딥러닝의 심장.